# Stacks and Queues

LIFO vs. FIFO: building both from scratch, common interview patterns (valid parentheses, monotonic stack, BFS with a queue), and how they compare to what we've already built (arrays, linked lists).

## Stacks vs. queues — same restriction, opposite order

Both are **restricted-access** structures: unlike an array or a dict, you don't get to touch an arbitrary element — you can only interact with one end (or, for a queue, one end each).

- **Stack — LIFO (last in, first out).** Think a stack of plates: you only ever add to or remove from the top. Core operations: `push` (add to top), `pop` (remove from top), `peek` (look at the top without removing it).
- **Queue — FIFO (first in, first out).** Think a line at a store: whoever arrived first gets served first. Core operations: `enqueue` (add to the back), `dequeue` (remove from the front), `peek` (look at the front without removing it).

**Where they show up:** stacks are behind undo/redo, call stacks (function calls), matching/balancing problems (parentheses, backtracking), and the monotonic-stack pattern. Queues are behind BFS, task scheduling, and anything modeling "process in the order things arrived."

**Implementation choices:** both can be built on top of either an array (Python `list`) or a linked list — each has a natural fit and a gotcha, which is worth working through hands-on rather than just stating.

## Stack / queue time complexity

| Operation | Complexity | Why |
|---|---|---|
| Push / pop / enqueue / dequeue | O(1) | Only ever touches one end (or, for a queue, one designated end each) — no shifting or scanning |
| Peek | O(1) | Just reads the top/front element without removing it |
| Lookup (arbitrary element, not the top/front) | O(n) | Not part of the interface — the only way to reach it is to remove everything in front of it first (destructively, unless you're walking a linked list) |

**Gotcha: O(1) isn't automatic — it depends on the implementation.** If a queue is backed by a plain Python list and `dequeue` is implemented as `list.pop(0)`, that's actually **O(n)**, not O(1) — removing the first element means shifting every remaining element down by one, the exact same problem as array-prepend. A true O(1) queue needs either `collections.deque`, a circular buffer, or a linked list with **both** a head and a tail pointer, so both ends can be touched without shifting anything.

In [ ]:
# TODO: Implement a Stack class backed by a Python list.
#
# Support:
# - push(value)  -> add to the top
# - pop()        -> remove and return the top element
# - peek()       -> return the top element without removing it
# - is_empty()   -> True/False

# Worst-case time: O(1) (amortized) for all methods -- push/pop/peek only touch the end of the list
# Space: O(n) to store n elements; O(1) extra space per operation
class Stack:
    def __init__(self):
        self._items = []
    
    def push(self, value):
        self._items.append(value)
    
    def pop(self):
        if not self._items:
            raise ValueError("nothing to pop!")
        return self._items.pop()
    
    def is_empty(self):
        return not self._items
    
    def peek(self):
        if not self._items:
            raise ValueError("nothing to see!")
        return self._items[-1]

a = Stack()
#print(a.is_empty())
a.push(1)
#print(a.is_empty())
a.push(2)
#print(a.peek())
a.pop()

In [ ]:
# TODO: Implement a Stack class backed by a linked list (build your own Node).
#
# Support:
# - push(value)  -> add to the top
# - pop()        -> remove and return the top element
# - peek()       -> return the top element without removing it
# - is_empty()   -> True/False

### first import my implementation of a singly linkedlist
class SinglyNode:
    def __init__(self, value):
        self.value = value
        self.next = None


class SinglyLinkedList:
    def __init__(self):
        self._head = None
        self._tail = None
        self._length = 0
    
    def append(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._tail
            self._tail = SinglyNode(value)
            _current.next = self._tail
        
        self._length += 1
    

    def prepend(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._head
            self._head = SinglyNode(value)
            self._head.next = _current
        
        self._length += 1
    
    def delete_at(self, index):
        
        if index >= self._length or index <0: 
            raise ValueError("index value is out of bound")

        dummy = SinglyNode(None)
        dummy.next = self._head
        node = dummy

        for i in range(index):
            node = node.next
        
        node.next = node.next.next
        self._length -= 1
        self._head = dummy.next
        if not node.next:
            self._tail = node if node is not dummy else None

    
### now build the stack

# Worst-case time: O(1) for all methods -- push/pop/peek only touch the head, no traversal needed
# Space: O(n) to store n elements; O(1) extra space per operation
class Stack:
    def __init__(self):
       self._items = SinglyLinkedList()
    
    def push(self, value):
        self._items.prepend(value)
    
    def pop(self):
        if not self._items._head:
            raise ValueError("nothing to pop!")
        value = self._items._head.value
        self._items.delete_at(0)
        return value
    
    def is_empty(self):
        return self._items._length == 0
    
    def peek(self):
        if not self._items._head:
            raise ValueError("nothing to see!")
        return self._items._head.value


a = Stack()
print(a.is_empty())
a.push(1)
print(a.is_empty())
a.push(2)
print(a.peek())
a.pop()

## Array-backed vs. linked-list-backed stack — same Big-O, different reason

Both implementations above are O(1) for `push`/`pop`/`peek`, but they get there differently:

- **Array-backed:** operates on the **end** of the list (`append`/`pop()` with no index). Python lists are dynamic arrays with spare capacity at the end, so adding/removing there needs no shifting — amortized O(1).
- **Linked-list-backed:** operates on the **head**, not the tail. A singly linked list only has `next` pointers, so removing the tail would require finding its predecessor by walking from `_head` — O(n). Removing the head needs no predecessor lookup at all, so head-based push/pop is O(1) regardless of list length.

Same complexity, opposite ends — the array's "cheap end" is the tail; the singly linked list's "cheap end" is the head.

**In practice, array-backed is usually still faster.** Same Big-O, but different constant factors: array elements sit contiguously in memory, so access/push benefits from CPU cache locality. Linked-list nodes are separate heap allocations that can be scattered in memory, so traversal means "pointer chasing" — more cache misses — plus each node pays extra memory for the `next` pointer. This is also why `collections.deque` (Python's real-world choice) is block-based rather than a plain linked list, to get O(1) at both ends *and* keep good cache locality.

In [ ]:
# TODO: Implement a Queue class backed by a Python list.
#
# Support:
# - enqueue(value) -> add to the back
# - dequeue()      -> remove and return the front element
# - peek()         -> return the front element without removing it
# - is_empty()     -> True/False

# Worst-case time: O(n) -- dequeue shifts every remaining element down after removing index 0
# Space: O(n) to store n elements; O(1) extra space per operation (in-place del)
class Queue:
    def __init__(self):
        self._items = []
    
    def enqueue(self, value):
        self._items.append(value)
    
    def dequeue(self):
        if not self._items:
            raise ValueError("nothing to pop!")
        value = self._items[0]
        del self._items[0] # o(n)!!!
        return value
    
    def is_empty(self):
        return not self._items
    
    def peek(self):
        if not self._items:
            raise ValueError("nothing to see!")
        return self._items[0]

a = Queue()
print(a.is_empty())
a.push(1)
print(a.is_empty())
a.push(2)
print(a.peek())
a.pop()

In [ ]:
# TODO: Implement a Queue class backed by a linked list (build your own Node).
#
# Support:
# - enqueue(value) -> add to the back
# - dequeue()      -> remove and return the front element
# - peek()         -> return the front element without removing it
# - is_empty()     -> True/False

### first import my implementation of a singly linkedlist
class SinglyNode:
    def __init__(self, value):
        self.value = value
        self.next = None


class SinglyLinkedList:
    def __init__(self):
        self._head = None
        self._tail = None
        self._length = 0
    
    def append(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._tail
            self._tail = SinglyNode(value)
            _current.next = self._tail
        
        self._length += 1
    

    def prepend(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._head
            self._head = SinglyNode(value)
            self._head.next = _current
        
        self._length += 1
    
    def delete_at(self, index):
        
        if index >= self._length or index <0: 
            raise ValueError("index value is out of bound")

        dummy = SinglyNode(None)
        dummy.next = self._head
        node = dummy

        for i in range(index):
            node = node.next
        
        node.next = node.next.next
        self._length -= 1
        self._head = dummy.next
        if not node.next:
            self._tail = node if node is not dummy else None

    
### now build the stack
# Worst-case time: O(1) for all methods -- enqueue touches the tail, dequeue/peek touch the head, no traversal needed
# Space: O(n) to store n elements; O(1) extra space per operation

class Queue:
    def __init__(self):
       self._items = SinglyLinkedList()
    
    def push(self, value):
        self._items.append(value)
    
    def pop(self):
        if not self._items._head:
            raise ValueError("nothing to pop!")
        value = self._items._head.value
        self._items.delete_at(0)
        return value
    
    def is_empty(self):
        return self._items._length == 0
    
    def peek(self):
        if not self._items._head:
            raise ValueError("nothing to see!")
        return self._items._head.value


a = Queue()
print(a.is_empty())
a.push(1)
a.push(3)
print(a.is_empty())
a.push(2)
print(a.peek())
a.pop()


## Array-backed vs. linked-list-backed queue — this time the Big-O actually differs

Unlike the stack case, the two queue implementations are **not** equally good:

- **Array-backed:** `enqueue` (append to end) is O(1) amortized, but `dequeue` (remove from front) is O(n) — removing index 0 shifts every remaining element down. There's no way around this with a plain Python list; the front is fundamentally the array's "expensive end."
- **Linked-list-backed:** `enqueue` appends at the tail (O(1), since `_tail` is stored directly), and `dequeue` removes the head (O(1), no predecessor lookup needed, unlike the stack's tail-removal problem). Both ends are cheap because insertion and removal happen at *opposite* ends of the list.

This is the same point cell-2's gotcha note made: a queue backed by an array (without a deque/circular buffer trick) is stuck with an O(n) end; a singly linked list with both `_head` and `_tail` pointers gets true O(1) on both operations.

In [48]:
# Interview problem: Implement Queue using Stacks
#
# Implement a first-in-first-out (FIFO) queue using only two stacks.
# The implemented queue should support all the functions of a normal queue
# (push, peek, pop, and empty):
#
# - void push(int x)  -> pushes element x to the back of the queue
# - int pop()         -> removes the element from the front of the queue and returns it
# - int peek()        -> returns the element at the front of the queue
# - boolean empty()   -> returns whether the queue is empty
#
# You must use only standard stack operations -- push to top, peek/pop from top,
# size, and is empty.

## first implement my own stack
### first import my implementation of a singly linkedlist
class SinglyNode:
    def __init__(self, value):
        self.value = value
        self.next = None


class SinglyLinkedList:
    def __init__(self):
        self._head = None
        self._tail = None
        self._length = 0
    
    def append(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._tail
            self._tail = SinglyNode(value)
            _current.next = self._tail
        
        self._length += 1
    

    def prepend(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._head
            self._head = SinglyNode(value)
            self._head.next = _current
        
        self._length += 1
    
    def delete_at(self, index):
        
        if index >= self._length or index <0: 
            raise ValueError("index value is out of bound")

        dummy = SinglyNode(None)
        dummy.next = self._head
        node = dummy

        for i in range(index):
            node = node.next
        
        node.next = node.next.next
        self._length -= 1
        self._head = dummy.next
        if not node.next:
            self._tail = node if node is not dummy else None

class Stack:
    def __init__(self):
       self._items = SinglyLinkedList()
    
    def push(self, value):
        self._items.prepend(value)
    
    def __len__(self):
        return self._items._length
    
    def pop(self):
        if not self._items._head:
            raise ValueError("nothing to pop!")
        value = self._items._head.value
        self._items.delete_at(0)
        return value
    
    def is_empty(self):
        return self._items._length == 0
    
    def peek(self):
        if not self._items._head:
            raise ValueError("nothing to see!")
        return self._items._head.value

# now we are building Queue on using two stacks

class Queue:
    def __init__(self):
        self._first = Stack()
        
    def push(self, value):
        self._first.push(value)
        
    def pop(self):
        if len(self._first)==0:
            raise ValueError("nothing to pop!")
        
        self._second = Stack()
        for _ in range(len(self._first)):
            self._second.push(self._first.pop())
        value = self._second.pop()

        self._first = Stack()
        for _ in range(len(self._second)):
            self._first.push(self._second.pop())

        return value
    
    def is_empty(self):
        return len(self._first) == 0
    
    def peek(self):
        if len(self._first)==0:
            raise ValueError("nothing to see!")
        
        self._second = Stack()
        for _ in range(len(self._first)):
            self._second.push(self._first.pop())
        value = self._second.peek()

        self._first = Stack()
        for _ in range(len(self._second)):
            self._first.push(self._second.pop())

        return value

a = Queue()
#print(a.is_empty())
a.push(1)
a.push(3)
#print(a.is_empty())
a.push(2)
print(a.peek())
a.pop()
     

1


1